# 17. Гибрид: мета-модель как поправка к смешиванию (Kaggle)

В ноутбуке 16 мета-LightGBM, обученный с нуля, оказался хуже простого линейного смешивания трёх моделей.
Здесь деревья стартуют со смешивания (`--on-blend`) и учат только поправку: по горизонту, типу дня
и недавним ошибкам станции, соседей и города. Те же абляции, имена `hybr_*`.

**Input:** датасет `vkr-transit-data` и вывод ноутбука `hybrid` (16) — там уже есть все прогнозы, включая период stack.
Accelerator — None, Internet — On. Время ~1,5–2 ч. Результат — `results_csv.zip` в Output.

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## 1. Прогнозы из ноутбука 16

In [ ]:
import pandas as pd
for p in sorted(glob.glob("/kaggle/input/**/results/preds/*.npz", recursive=True)):
    dst = f"{R}/preds/{os.path.basename(p)}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
logs = glob.glob("/kaggle/input/**/results/runs.csv", recursive=True)
if logs:
    pd.concat([pd.read_csv(p) for p in logs]).drop_duplicates().to_csv(f"{R}/runs.csv", index=False)
for p in glob.glob("/kaggle/input/**/results/fi_*.csv", recursive=True):
    shutil.copy(p, R)
need = [f"{m}__{s}.npz" for m in ("lgbm_final", "gnn_od_in", "chronos2") for s in ("stack", "val", "test")]
missing = [n for n in need if not os.path.exists(f"{R}/preds/{n}")]
assert not missing, f"Нет прогнозов {missing}: подключите вывод ноутбука 16"

## 2. Гибрид-поправка

In [ ]:
t = time.time()
!cd {REPO} && python src/train_hybrid.py --on-blend --prefix hybr --variants full nonbr no_resid no_chronos no_gnn lgbm_only
print(f"гибрид: {(time.time() - t) / 60:.0f} мин")

## 3. Оценка

In [ ]:
!cd {REPO} && python src/evaluate.py --split val --ref norm_level

In [ ]:
!cd {REPO} && python src/evaluate.py --split test --ref norm_level

In [ ]:
PAIRS = " ".join([
    "hybr_full:hyb_blend", "hybr_full:hyb_full", "hyb_blend:lgbm", "hybr_full:lgbm",   # поправка к смешиванию
    "hybr_nonbr:hybr_full", "hybr_no_resid:hybr_full",                                 # соседи, недавние ошибки
    "hybr_no_chronos:hybr_full", "hybr_no_gnn:hybr_full", "hybr_lgbm_only:lgbm_final",
])
!cd {REPO} && python src/report.py --split val --pairs {PAIRS}

In [ ]:
!cd {REPO} && python src/report.py --split test --pairs {PAIRS}

In [ ]:
pd.set_option("display.width", 200, "display.max_columns", 20)
H_ = ["горизонт h1", "горизонт h2-3", "горизонт h4-6", "горизонт h7-12", "горизонт h13-24"]
for split in ["val", "test"]:
    m = pd.read_csv(f"{R}/metrics_{split}.csv")
    C = m[(m.subset == "все точки") & (m.model.str.startswith("hyb") | m.model.isin(["norm_level", "lgbm", "lgbm_final"]))]
    t = C[C.slice == "всё"].set_index("model")[["wape", "bias"]].join(
        C[C.slice.isin(H_)].pivot_table(index="model", columns="slice", values="wape")[H_])
    print(f"\n{split}"); display(t.round(4).sort_values("wape"))

## 4. Архив

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/*.csv"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*.csv")))